# Sustainability Reports Preprocessing for FinBERT Sentiment Analysis

## Objective

The objective of this notebook is to preprocess sustainability reports before performing sentiment analysis with FinBERT.

The preprocessing workflow includes extracting text from PDF reports, cleaning the extracted content, splitting each report into individual sentences using the Punkt sentence tokenizer, cleaning the resulting sentences, and saving each processed report as a separate structured file.

The generated files will serve as the input dataset for the subsequent FinBERT Sentiment Analysis notebook.

## 1. Install Required Packages

In [183]:
!pip install pdfplumber nltk openpyxl

## 2. Import Required Libraries

This section imports all the libraries required for PDF extraction, text preprocessing, sentence tokenization, data manipulation, and file management.

In [184]:
import os
import glob
import zipfile
import re

import pandas as pd
import pdfplumber


import nltk

from nltk.tokenize import sent_tokenize

## 3. Download the Punkt Tokenizer

In [185]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 4. Upload Sustainability Reports

This section uploads a compressed folder containing a sample of sustainability reports. The reports will be extracted and prepared for text preprocessing.

In [290]:
import os

pdf_files = []

for file in os.listdir("/content"):
    if file.lower().endswith(".pdf"):
        pdf_files.append(os.path.join("/content", file))

print(f"Found {len(pdf_files)} PDF reports:")
for pdf in pdf_files:
    print(os.path.basename(pdf))

Found 19 PDF reports:
sus report refresco.pdf
sus report leche pescual.pdf
trolli Code of Conduct Mederer Group (version 0425) - ENG.pdf
sus report grup apex.pdf
sus report idilia.pdf
sus report schreiber.pdf
sus report nestle.pdf
sus report lafuente.pdf
sus report grupo an.pdf
WKKC_SB Report_2025_11_19_25_FINAL.pdf
sus report liquats.pdf
sus report lactalis.pdf
sus report vicky foods.pdf
sus report sebater.pdf
sus report mandelez.pdf
sus report trolli.pdf
sus report procavi.pdf
sus report jealsa.pdf
sus report monells.pdf


## 5. Extract Text from Sustainability Reports

This section extracts the textual content from each sustainability report using the `pdfplumber` library. The extracted text is stored together with the report name in a structured DataFrame for further preprocessing.

In [291]:
documents = []

for pdf_path in pdf_files:

    company_name = os.path.splitext(os.path.basename(pdf_path))[0]

    report_text = ""

    print(f"Processing: {company_name}")

    try:
        doc = fitz.open(pdf_path)

        for page in doc:
            text = page.get_text()

            if text:
                report_text += text + "\n"

        doc.close()

        print(f"  Characters extracted: {len(report_text)}")

    except Exception as e:
        print(f"  ERROR: {e}")

    print("-" * 50)

    documents.append({
        "Company": company_name,
        "Text": report_text
    })

documents_df = pd.DataFrame(documents)

Processing: sus report refresco
  Characters extracted: 289886
--------------------------------------------------
Processing: sus report leche pescual
  Characters extracted: 170231
--------------------------------------------------
Processing: trolli Code of Conduct Mederer Group (version 0425) - ENG
  Characters extracted: 11433
--------------------------------------------------
Processing: sus report grup apex
  Characters extracted: 159106
--------------------------------------------------
Processing: sus report idilia
  Characters extracted: 5529
--------------------------------------------------
Processing: sus report schreiber
  Characters extracted: 37801
--------------------------------------------------
Processing: sus report nestle
  Characters extracted: 786656
--------------------------------------------------
Processing: sus report lafuente
  Characters extracted: 2685
--------------------------------------------------
Processing: sus report grupo an
  Characters extracte

## 6. Inspect the Extracted Text

This section displays the extracted text to verify that the PDF reports were successfully processed before applying text cleaning and sentence tokenization.

In [292]:
documents_df.head()

,Company,Text
0,sus report refresco,Annual Report\n2025\nPegasus MidCo B.V.\n\nCon...
1,sus report leche pescual,Corporación Empresarial Pascual\nESTADO DE LA ...
2,trolli Code of Conduct Mederer Group (version ...,Our code of conduct\nMederer Group\nApril 2025...
3,sus report grup apex,\n \n \n1 \n \n \n \n \n \n\n \n \n \n2 \n ...
4,sus report idilia,\n \nINFORMACIÓN CONFIDENCIAL | © IDILIA FOOD...


In [293]:
import os

for pdf in pdf_files:
    print(os.path.basename(pdf))
    print("Size:", os.path.getsize(pdf), "bytes")
    print()

sus report refresco.pdf
Size: 46684417 bytes

sus report leche pescual.pdf
Size: 24681055 bytes

trolli Code of Conduct Mederer Group (version 0425) - ENG.pdf
Size: 255666 bytes

sus report grup apex.pdf
Size: 4539383 bytes

sus report idilia.pdf
Size: 247892 bytes

sus report schreiber.pdf
Size: 13203628 bytes

sus report nestle.pdf
Size: 16822620 bytes

sus report lafuente.pdf
Size: 455034 bytes

sus report grupo an.pdf
Size: 47429912 bytes

WKKC_SB Report_2025_11_19_25_FINAL.pdf
Size: 12721100 bytes

sus report liquats.pdf
Size: 12349322 bytes

sus report lactalis.pdf
Size: 7515129 bytes

sus report vicky foods.pdf
Size: 5173784 bytes

sus report sebater.pdf
Size: 1705793 bytes

sus report mandelez.pdf
Size: 9903251 bytes

sus report trolli.pdf
Size: 255666 bytes

sus report procavi.pdf
Size: 5981409 bytes

sus report jealsa.pdf
Size: 4000535 bytes

sus report monells.pdf
Size: 17140302 bytes



In [294]:
import fitz
import os

for pdf_path in pdf_files:
    try:
        doc = fitz.open(pdf_path)
        print(f"{os.path.basename(pdf_path)} -> {len(doc)} pages")
        doc.close()
    except Exception as e:
        print(f"{os.path.basename(pdf_path)} -> ERROR: {e}")

sus report refresco.pdf -> 94 pages
sus report leche pescual.pdf -> 123 pages
trolli Code of Conduct Mederer Group (version 0425) - ENG.pdf -> 22 pages
sus report grup apex.pdf -> 78 pages
sus report idilia.pdf -> 2 pages
sus report schreiber.pdf -> 64 pages
sus report nestle.pdf -> 253 pages
sus report lafuente.pdf -> 1 pages
sus report grupo an.pdf -> 152 pages
WKKC_SB Report_2025_11_19_25_FINAL.pdf -> 32 pages
sus report liquats.pdf -> 72 pages
sus report lactalis.pdf -> 51 pages
sus report vicky foods.pdf -> 10 pages
sus report sebater.pdf -> 17 pages
sus report mandelez.pdf -> 72 pages
sus report trolli.pdf -> 22 pages
sus report procavi.pdf -> 88 pages
sus report jealsa.pdf -> 84 pages
sus report monells.pdf -> 59 pages


In [295]:
documents_df["Text_Length"] = documents_df["Text"].str.len()

documents_df[["Company", "Text_Length"]]

,Company,Text_Length
0,sus report refresco,289886
1,sus report leche pescual,170231
2,trolli Code of Conduct Mederer Group (version ...,11433
3,sus report grup apex,159106
4,sus report idilia,5529
5,sus report schreiber,37801
6,sus report nestle,786656
7,sus report lafuente,2685
8,sus report grupo an,157609
9,WKKC_SB Report_2025_11_19_25_FINAL,45207


In [296]:
documents_df.loc[0, "Text"][:1000]

'Annual Report\n2025\nPegasus MidCo B.V.\n\nContents\nExecutive Board Report\n3\nMessage from the CEO\n4\n2025 Strategic review\n6\nFinancial results\n15\nRisks and risk management\n20\nFinancial statements 2025\n26\nConsolidated income statement\n27\nConsolidated statement of other comprehensive income\n28\nConsolidated balance sheet\n29\nConsolidated statement of changes in equity\n30\nConsolidated statement of cash flows\n31\nNotes to the consolidated financial statements\n32\nCompany income statement\n79\nCompany balance sheet\n80\nNotes to the company financial statements\n81\nOther information\n84\nStatutory provision with respect to appropriation\nof result\n85\nIndependent auditor’s report\n86\nAppendices\n89\nContact us\n90\nGlossary\n92\nForward-looking statements\n93\nIntroduction\n|\nExecutive Board report\n|\nFinancial statements 2025\n|\nOther information\n|\nAdditional information\nRefresco Annual Report 2025\n2\n\nExecutive Board Report\nMessage from the CEO\n2025 strat

## 7. Normalize the Extracted Text

This section performs a light normalization of the extracted text by removing unnecessary line breaks, tabs, and repeated spaces while preserving the original content. This normalization improves sentence segmentation using the Punkt tokenizer.

In [297]:
import re

def normalize_text(text):

    if pd.isna(text):
        return ""

    # Replace line breaks with spaces
    text = text.replace("\n", " ")

    # Replace tabs with spaces
    text = text.replace("\t", " ")

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

documents_df["Normalized_Text"] = documents_df["Text"].apply(normalize_text)

print("Text normalization completed successfully.")

Text normalization completed successfully.


## 8. Split Reports into Sentences

This section segments each normalized sustainability report into individual sentences using the Punkt sentence tokenizer. Each sentence will later be cleaned, analyzed, and stored separately.

In [298]:
documents_df["Sentences"] = documents_df["Normalized_Text"].apply(sent_tokenize)

print("Sentence tokenization completed successfully.")

Sentence tokenization completed successfully.


## 9. Inspect the Tokenized Sentences

This section displays a sample of the tokenized sentences to verify that the reports have been correctly segmented before applying sentence-level preprocessing.

In [299]:
company = documents_df.loc[0, "Company"]

print(f"Company: {company}\n")

for i, sentence in enumerate(documents_df.loc[0, "Sentences"][:10], start=1):
    print(f"Sentence {i}:")
    print(sentence)
    print("-" * 80)

Company: sus report refresco

Sentence 1:
Annual Report 2025 Pegasus MidCo B.V.
--------------------------------------------------------------------------------
Sentence 2:
Contents Executive Board Report 3 Message from the CEO 4 2025 Strategic review 6 Financial results 15 Risks and risk management 20 Financial statements 2025 26 Consolidated income statement 27 Consolidated statement of other comprehensive income 28 Consolidated balance sheet 29 Consolidated statement of changes in equity 30 Consolidated statement of cash flows 31 Notes to the consolidated financial statements 32 Company income statement 79 Company balance sheet 80 Notes to the company financial statements 81 Other information 84 Statutory provision with respect to appropriation of result 85 Independent auditor’s report 86 Appendices 89 Contact us 90 Glossary 92 Forward-looking statements 93 Introduction | Executive Board report | Financial statements 2025 | Other information | Additional information Refresco Annual 

## 10. Create the Sentence-Level Dataset

This section creates a sentence-level dataset for each sustainability report. Each sentence is stored in its original form together with a cleaned version and its corresponding word count.

In [300]:
import re

def clean_sentence(sentence):
    """
    Clean a sentence while preserving meaningful words.
    """

    # Convert to lowercase
    sentence = sentence.lower()

    # Remove line breaks
    sentence = sentence.replace("\n", " ")

    # Remove punctuation
    sentence = re.sub(r"[^\w\s]", " ", sentence)

    # Remove extra spaces
    sentence = re.sub(r"\s+", " ", sentence)

    return sentence.strip()


sentence_data = []

for _, row in documents_df.iterrows():

    company = row["Company"]

    for i, sentence in enumerate(row["Sentences"], start=1):

        cleaned = clean_sentence(sentence)

        word_count = len(cleaned.split())

        sentence_data.append({

            "Company": company,

            "Sentence_ID": i,

            "Original_Sentence": sentence,

            "Cleaned_Sentence": cleaned,

            "Word_Count": word_count

        })

sentences_df = pd.DataFrame(sentence_data)

print(f"Total sentences extracted: {len(sentences_df)}")

Total sentences extracted: 12583


## 11. Inspect the Sentence-Level Dataset

This section displays the sentence-level dataset after preprocessing.

In [301]:
sentences_df.head(10)

,Company,Sentence_ID,Original_Sentence,Cleaned_Sentence,Word_Count
0,sus report refresco,1,Annual Report 2025 Pegasus MidCo B.V.,annual report 2025 pegasus midco b v,7
1,sus report refresco,2,Contents Executive Board Report 3 Message from...,contents executive board report 3 message from...,191
2,sus report refresco,3,"Today, we stand on a strong foundation of scal...",today we stand on a strong foundation of scale...,11
3,sus report refresco,4,I stepped into the role of Chief Executive Off...,i stepped into the role of chief executive off...,28
4,sus report refresco,5,This year was one of ambition amid market vola...,this year was one of ambition amid market vola...,9
5,sus report refresco,6,"While further diversifying our business, we al...",while further diversifying our business we als...,19
6,sus report refresco,7,"Looking ahead, our opportunity lies in unlocki...",looking ahead our opportunity lies in unlockin...,22
7,sus report refresco,8,"Growth is in our DNA, and with energized teams...",growth is in our dna and with energized teams ...,24
8,sus report refresco,9,I am excited about what we will build together.,i am excited about what we will build together,9
9,sus report refresco,10,Year in Review 2025 was marked by continued ma...,year in review 2025 was marked by continued ma...,15


In [302]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

def remove_illegal_characters(text):
    if pd.isna(text):
        return text
    return ILLEGAL_CHARACTERS_RE.sub("", str(text))

In [303]:
text_columns = [
    "Original_Sentence",
    "Cleaned_Sentence"
]

for col in text_columns:
    sentences_df[col] = sentences_df[col].apply(remove_illegal_characters)

## 12. Save the Processed Reports

This section saves each processed sustainability report as a separate Excel file. The generated files will be used as the input dataset for the FinBERT sentiment analysis notebook.

In [304]:
import os

output_folder = "Processed_Reports"
os.makedirs(output_folder, exist_ok=True)

for company, group in sentences_df.groupby("Company"):

    output_path = os.path.join(output_folder, f"{company}.xlsx")

    group.to_excel(output_path, index=False)

print("All reports have been saved successfully.")

All reports have been saved successfully.


## 13. Verify the Saved Reports

This section verifies that all processed sustainability reports have been successfully exported.

In [305]:
import os

saved_files = os.listdir(output_folder)

print(f"Number of reports: {len(saved_files)}\n")

for file in sorted(saved_files):
    print(file)

Number of reports: 82

.ipynb_checkpoints
Puratos-2025-Sustainability-GRI-Report.xlsx
Sarga Memoria-Sostenibilidad-web.xlsx
WKKC_SB Report_2025_11_19_25_FINAL.xlsx
elpozo-cuadriptico-ing-2024.xlsx
pepsico 2025-ESG-Performance-Metrics-and-Calculation-Methodology.xlsx
sus plan 2030 grupoapex.xlsx
sus rapport comapny brokaw.xlsx
sus rapport company AGROMILLORA.xlsx
sus rapport company BARBERET.xlsx
sus rapport company BONNY SOCIEDAD.xlsx
sus rapport company BORGES AGRICULTURAL.xlsx
sus rapport company HERNANDEZ.xlsx
sus rapport company LA FORJA SELECCION SOCIEDAD.xlsx
sus rapport company cefusa.xlsx
sus rapport company costa brava.xlsx
sus rapport company el limonar.xlsx
sus rapport company padesa.xlsx
sus rapport company planasa.xlsx
sus rapport company plukon.xlsx
sus rapport company uvesa.xlsx
sus rapport company vall companys.xlsx
sus rapport palacios.xlsx
sus report acor.xlsx
sus report agrosevilla.xlsx
sus report aira.xlsx
sus report aldelis.xlsx
sus report angel camacho.xlsx
sus re

## 14. Download the Processed Reports

Google Colab stores generated files in temporary storage. This section compresses all processed reports into a ZIP archive and downloads it to the local computer.

In [306]:
import shutil

# Create ZIP archive
shutil.make_archive("Processed_Reports", "zip", output_folder)

print("ZIP file created successfully.")

ZIP file created successfully.


In [307]:
from google.colab import files

files.download("Processed_Reports.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>